In [1]:
!pip install transformers datasets torch scikit-learn


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from datasets import load_dataset

dataset = load_dataset("ag_news")

C:\Users\Asus GL502-VM\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Asus GL502-VM\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Asus GL502-VM\.cache\huggingface\hub\datasets--ag_news. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Pyth

In [3]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})


In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [6]:
def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding="max_length")

In [7]:
dataset = dataset.map(tokenize)

Map: 100%|████████████████████████████████████████████████████████████████| 7600/7600 [00:05<00:00, 1391.30 examples/s]


In [8]:
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

In [9]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=4
)

C:\Users\Asus GL502-VM\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Asus GL502-VM\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|███████████████████████████████████████████████████████████

In [10]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=8,
    num_train_epochs=1,
    logging_steps=50
)

ImportError: Using the `Trainer` with `PyTorch` requires `accelerate>=1.1.0`: Please run `pip install transformers[torch]` or `pip install 'accelerate>=1.1.0'`

In [11]:
!pip install accelerate

   ---------------------------------------- 0.0/383.7 kB ? eta -:--:--
   - -------------------------------------- 10.2/383.7 kB ? eta -:--:--
   - -------------------------------------- 10.2/383.7 kB ? eta -:--:--
   --- ----------------------------------- 30.7/383.7 kB 325.1 kB/s eta 0:00:02
   ---- ---------------------------------- 41.0/383.7 kB 281.8 kB/s eta 0:00:02
   --------- ----------------------------- 92.2/383.7 kB 476.3 kB/s eta 0:00:01
   --------------- ---------------------- 153.6/383.7 kB 612.6 kB/s eta 0:00:01
   -------------------- ----------------- 204.8/383.7 kB 731.4 kB/s eta 0:00:01
   ------------------------------ ------- 307.2/383.7 kB 905.4 kB/s eta 0:00:01
   ---------------------------------------- 383.7/383.7 kB 1.1 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
import sys
!"{sys.executable}" -m pip install accelerate


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
from torch.utils.data import DataLoader

train_loader = DataLoader(dataset["train"].select(range(2000)), batch_size=8)

In [17]:
import torch
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [18]:
model.train()

for epoch in range(1):   # 1 epoch
    for batch in train_loader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        outputs = model(
            input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()

    print("Epoch done, Loss:", loss.item())

KeyboardInterrupt: 

In [ ]:
print("Epoch done, Loss:", loss.item())

In [19]:
train_loader = DataLoader(dataset["train"].select(range(500)), batch_size=8)

In [20]:
model.eval()

text = "Apple launches new AI technology"

inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)

with torch.no_grad():
    outputs = model(**inputs)

pred = outputs.logits.argmax().item()

labels = ["World", "Sports", "Business", "Sci/Tech"]

print("Prediction:", labels[pred])

Prediction: Sci/Tech
